# CoT Analysis for Dynamic NIAH

This notebook runs thinking/non-thinking response generation, full-sequence outlier-token analysis, and prompt-vs-generation attention diagnostics for Dynamic NIAH tasks. Core logic lives in `src/counting/cot_analysis.py` and `scripts/run_cot_analysis.py`.

## 1. Mount Drive and choose the repo

In [ ]:
from google.colab import drive
from pathlib import Path
import os
import sys

drive.mount('/content/drive')

# CHANGE THIS to your checked-out dataset-generation repository.
REPO_DIR = Path('/content/drive/MyDrive/Colab Notebooks/compression/dataset-generation-main-v15')
if not REPO_DIR.exists():
    raise FileNotFoundError(
        f'REPO_DIR does not exist: {REPO_DIR}. '
        'Update REPO_DIR to your dataset-generation checkout before continuing.'
    )

os.chdir(REPO_DIR)
if str(REPO_DIR / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_DIR / 'src'))

print('Working directory:', Path.cwd())

## 2. Install dependencies

In [ ]:
!pip -q install -U transformers accelerate datasets sentencepiece safetensors huggingface-hub tqdm matplotlib psutil

## 3. Global Configuration

In [ ]:
from pathlib import Path

# Dataset settings.
TASK_TYPE = 'match_count'  # Dynamic NIAH task; match_count is the default smoke target.
MODEL_NAME = 'Qwen/Qwen3-8B'  # Model used for generation and analysis.
TOKENIZER_NAME = None  # Defaults to MODEL_NAME when None.
NUM_EXAMPLES = 100  # Number of NIAH examples to generate or reuse.
TARGET_HAYSTACK_TOKENS = 1000  # Approximate prompt haystack length.
NUM_NEEDLES = 10  # Fixed needle count when NUM_MAX_NEEDLES is None.
NUM_MAX_NEEDLES = 10  # If positive, sample each example count uniformly from 1..NUM_MAX_NEEDLES.
INSERTION_POSITIONS = [0] * 10  # Passed to the data generator; ignored when randomized.
RANDOMIZE_NEEDLE_INSERTION = True  # Randomize needle positions.
RANDOMIZE_NEEDLE_SEED = 42  # Seed for randomized insertion.
SENTENCE_LEVEL_INSERTION = True  # Sentence-boundary insertion strategy.
WORD_LEVEL_INSERTION = False  # Word-boundary insertion strategy.
GLOBAL_RANDOM_SEED = 42  # Global seed recorded in config.
HAYSTACK_SEED = 123  # Haystack seed.
NEEDLE_SEED = 456  # Needle-content seed.
FACT_TEMPLATES_PATH = 'data/templates/niah_fact_single_template.txt'  # Fact template file.
COUNTING_NEEDLE_KIND = 'city_score'  # Needle kind for counting tasks.
MARKER_TEXT = '[dolphin]'  # Literal marker text when marker needles are used.
UID_TOKEN_LENGTH = 4  # Target UID token length for literal marker generation.
PROMPT_STYLE = 'vanilla'  # Prompt style.

# Generation settings.
THINKING_MODES = ['nonthinking', 'thinking']  # Run one or both modes on the same dataset rows.
MAX_NEW_TOKENS_NONTHINKING = 64  # Non-thinking generation budget.
MAX_NEW_TOKENS_THINKING = 1024  # Thinking generation budget.
GENERATION_TEMPERATURE = 0.0  # Keep deterministic unless deliberately sampling.
GENERATION_DO_SAMPLE = None  # None lets temperature decide sampling behavior.
GENERATION_TOP_P = None  # Optional top-p when sampling.
USE_KV_CACHE_FOR_NONTHINKING = True  # Preserve existing behavior by default.
USE_KV_CACHE_FOR_THINKING = True  # Preserve existing behavior by default.

# Outlier / attention analysis settings.
LAYERS = [4, 8, 12, 16, 20, 24, 28]  # Layers for hidden states and Q/K attention analysis.
K = 16  # Top-K attention sinks and massive-activation tokens per example/layer.
OUTLIER_RATIO_THRESHOLD = 5.0  # Only outlier tokens with score at least this value become attention patterns/plots.
MAX_ANALYSIS_EXAMPLES = 10  # Expensive analysis subset size.
ANALYSIS_EXAMPLE_SELECTION = 'balanced_success_failure'  # Prefer success/failure and count balance.
ATTENTION_HEAD_AGG = 'max_across_heads'  # Save per-head tables; plot max across heads.
SAVE_PER_HEAD_ATTENTION_TABLES = True  # Keep per-head attention statistics tables.
SAVE_QK_CACHE = False  # Temporary Q/K cache is allowed but cleaned before final zipping.
SAVE_FULL_HIDDEN_STATES = False  # Keep derived outputs; large tensors are cleaned before archive.
ANALYSIS_DTYPE = 'bfloat16_if_cuda_else_float32'  # Forward dtype for hidden-state capture.
PLOT_SPECIAL_TOKEN_LINES = True  # Draw thinking/final-answer dashed lines when detected.
THINKING_MARKER_STRINGS = ['</think>']  # Qwen-style thinking delimiter heuristic.
FINAL_ANSWER_MARKER_STRINGS = []  # Reserved for future task-specific answer markers.
CAPTURE_ATTN_IMPLEMENTATION = 'sdpa'  # Q/K capture attention implementation.
CAPTURE_MODEL_DTYPE = 'bf16'  # Model dtype for Q/K capture.
CAPTURE_SAVE_DTYPE = 'bf16'  # Temporary Q/K tensor save dtype.
QK_COMPUTE_DTYPE = 'fp32'  # Attention-stat compute dtype.
QK_KEY_BLOCK_SIZE = 8192  # Block size for Q/K attention calculations.
QK_QUERY_BLOCK_SIZE = 64  # Query block size for Q/K attention calculations.

# Stage switches.
RUN_DATASET_GENERATION = True  # Generate/reuse dataset.
RUN_RESPONSE_GENERATION = True  # Generate/reuse responses and metrics.
RUN_OUTLIER_ANALYSIS = True  # Run full-sequence outlier analysis.
RUN_ATTENTION_ANALYSIS = True  # Generate CoT attention tables.
RUN_PLOTTING = True  # Generate CoT attention plots.

# I/O settings.
USER_RUN_NAME = None  # Optional fixed run name.
RUN_ROOT = '/content'  # Local Colab runtime root.
DATA_CACHE_ROOT = 'data/niah-example'  # Reusable dataset/response cache root.
RESULTS_PATH = 'results/cot_analysis'  # Destination for final archive.
FORCE_REGENERATE_DATASET = False  # Ignore cached generated dataset.
FORCE_REGENERATE_RESPONSES = False  # Ignore cached response files.
ZIP_RESULTS_AFTER_RESPONSE_STAGE = True  # Checkpoint expensive response generation.
ZIP_RESULTS_AFTER_ANALYSIS_STAGE = True  # Archive final outputs.
LOG_GPU_MEMORY = True  # Record GPU memory in timing logs.
LOG_CPU_MEMORY = True  # Reserved for CPU memory logging.

## 4. Resolve Config and Paths

In [ ]:
import json
import os
import torch
from pathlib import Path

from transformers import AutoModelForCausalLM, AutoTokenizer

from counting.cot_analysis import (
    archive_cot_results,
    build_cot_analysis_run_config,
    cache_mode_outputs,
    collect_analysis_needle_span_eligibility,
    cleanup_cot_mode_artifacts,
    create_cot_run_paths,
    ensure_selected_full_sequence_artifacts,
    generate_or_load_dataset,
    plot_cot_attention_tables,
    restore_mode_outputs_from_cache,
    run_generation_for_mode,
    run_mode_qk_outlier_analysis,
    save_selected_hidden_states_for_examples,
    select_analysis_example_ids,
    write_cot_attention_projection_tables,
    write_cot_response_checkpoint_archive,
    write_selected_mode_run,
)
from counting.feature_analysis import StageTimer, print_gpu_memory_snapshot, release_torch_memory

CONFIG_KEYS = [
    'TASK_TYPE', 'MODEL_NAME', 'TOKENIZER_NAME', 'NUM_EXAMPLES', 'TARGET_HAYSTACK_TOKENS',
    'NUM_NEEDLES', 'NUM_MAX_NEEDLES', 'INSERTION_POSITIONS', 'RANDOMIZE_NEEDLE_INSERTION',
    'RANDOMIZE_NEEDLE_SEED', 'SENTENCE_LEVEL_INSERTION', 'WORD_LEVEL_INSERTION',
    'GLOBAL_RANDOM_SEED', 'HAYSTACK_SEED', 'NEEDLE_SEED', 'FACT_TEMPLATES_PATH',
    'COUNTING_NEEDLE_KIND', 'MARKER_TEXT', 'UID_TOKEN_LENGTH', 'PROMPT_STYLE',
    'THINKING_MODES', 'MAX_NEW_TOKENS_NONTHINKING', 'MAX_NEW_TOKENS_THINKING',
    'GENERATION_TEMPERATURE', 'GENERATION_DO_SAMPLE', 'GENERATION_TOP_P',
    'USE_KV_CACHE_FOR_NONTHINKING', 'USE_KV_CACHE_FOR_THINKING', 'LAYERS', 'K',
    'OUTLIER_RATIO_THRESHOLD', 'MAX_ANALYSIS_EXAMPLES', 'ANALYSIS_EXAMPLE_SELECTION', 'ATTENTION_HEAD_AGG',
    'SAVE_PER_HEAD_ATTENTION_TABLES', 'SAVE_QK_CACHE', 'SAVE_FULL_HIDDEN_STATES',
    'ANALYSIS_DTYPE', 'PLOT_SPECIAL_TOKEN_LINES', 'THINKING_MARKER_STRINGS',
    'FINAL_ANSWER_MARKER_STRINGS', 'CAPTURE_ATTN_IMPLEMENTATION', 'CAPTURE_MODEL_DTYPE',
    'CAPTURE_SAVE_DTYPE', 'QK_COMPUTE_DTYPE', 'QK_KEY_BLOCK_SIZE', 'QK_QUERY_BLOCK_SIZE',
    'RUN_DATASET_GENERATION', 'RUN_RESPONSE_GENERATION', 'RUN_OUTLIER_ANALYSIS',
    'RUN_ATTENTION_ANALYSIS', 'RUN_PLOTTING', 'USER_RUN_NAME', 'RUN_ROOT',
    'DATA_CACHE_ROOT', 'RESULTS_PATH', 'FORCE_REGENERATE_DATASET',
    'FORCE_REGENERATE_RESPONSES', 'ZIP_RESULTS_AFTER_RESPONSE_STAGE',
    'ZIP_RESULTS_AFTER_ANALYSIS_STAGE', 'LOG_GPU_MEMORY', 'LOG_CPU_MEMORY',
]
CONFIG = {name: globals()[name] for name in CONFIG_KEYS if name in globals()}
config_path = Path('/content/cot_analysis_config.json')
config_path.write_text(json.dumps({'config': CONFIG}, indent=2, ensure_ascii=False), encoding='utf-8')
print('Config path:', config_path)

cfg = build_cot_analysis_run_config(CONFIG)
paths = create_cot_run_paths(cfg)
timer = StageTimer(json_path=paths.timing_json_path, csv_path=paths.timing_csv_path)

def resolve_analysis_dtype():
    value = str(cfg['ANALYSIS_DTYPE'])
    if value == 'bfloat16_if_cuda_else_float32':
        return torch.bfloat16 if torch.cuda.is_available() else torch.float32
    return getattr(torch, value)

def load_model_and_tokenizer(attn_implementation=None):
    tokenizer = AutoTokenizer.from_pretrained(cfg['TOKENIZER_NAME'], trust_remote_code=True)
    kwargs = {
        'trust_remote_code': True,
        'device_map': 'auto',
        'torch_dtype': resolve_analysis_dtype(),
    }
    if attn_implementation is not None:
        kwargs['attn_implementation'] = attn_implementation
    model = AutoModelForCausalLM.from_pretrained(cfg['MODEL_NAME'], **kwargs)
    model.eval()
    return model, tokenizer

def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding='utf-8').splitlines() if line.strip()]

print('Run name:', paths.run_name)
print('Run dir:', paths.run_dir)
print('Stable setting name:', paths.setting_name)
print('Reusable data cache:', paths.cache_dir)
print('Modes:', cfg['THINKING_MODES'])
print('Resolved config:', paths.config_path)
print('Timing log:', paths.timing_json_path)

## 5. Dataset Generation or Reuse

In [ ]:
with timer.stage('dataset_generation_or_reuse'):
    rows = generate_or_load_dataset(
        cfg,
        paths,
        force=bool(cfg['FORCE_REGENERATE_DATASET']),
    )
print('Dataset rows:', len(rows))
print('Dataset path:', paths.dataset_path)

## 6. Response Generation and Metrics

In [ ]:
results_by_mode = {}
metrics_by_mode = {}
modes_to_generate = []

if cfg['RUN_RESPONSE_GENERATION']:
    with timer.stage('response_generation_and_scoring'):
        for mode in cfg['THINKING_MODES']:
            if (
                not paths.predictions_path(mode).exists()
                or not paths.metrics_path(mode).exists()
            ) and not cfg['FORCE_REGENERATE_RESPONSES']:
                if restore_mode_outputs_from_cache(cfg=cfg, paths=paths, mode=mode):
                    print(f'Restored cached responses for mode={mode} from data cache:', paths.cache_dir)

            if (
                paths.predictions_path(mode).exists()
                and paths.metrics_path(mode).exists()
                and not cfg['FORCE_REGENERATE_RESPONSES']
            ):
                print(f'Using existing responses for mode={mode}:', paths.predictions_path(mode))
                results_by_mode[mode] = read_jsonl(paths.predictions_path(mode))
                metrics_by_mode[mode] = json.loads(paths.metrics_path(mode).read_text(encoding='utf-8'))
            else:
                modes_to_generate.append(mode)

        print('Modes needing generation:', modes_to_generate)
        if modes_to_generate:
            model, tokenizer = load_model_and_tokenizer()
            print_gpu_memory_snapshot('cot-analysis after generation model load')
            try:
                for mode in modes_to_generate:
                    results, metrics = run_generation_for_mode(
                        cfg=cfg,
                        paths=paths,
                        mode=mode,
                        rows=rows,
                        model=model,
                        tokenizer=tokenizer,
                    )
                    results_by_mode[mode] = results
                    metrics_by_mode[mode] = metrics
                    cache_dir = cache_mode_outputs(cfg=cfg, paths=paths, mode=mode)
                    print(f'Cached mode={mode} outputs to {cache_dir}')
            finally:
                del model
                release_torch_memory(collect_garbage=True)
        else:
            print('All requested modes restored from run folder or stable data cache; model generation skipped.')
else:
    for mode in cfg['THINKING_MODES']:
        print(f'Loading existing responses for mode={mode}')
        results_by_mode[mode] = read_jsonl(paths.predictions_path(mode))
        metrics_by_mode[mode] = json.loads(paths.metrics_path(mode).read_text(encoding='utf-8'))

print(json.dumps(metrics_by_mode, indent=2, ensure_ascii=False))

## 7. Response Checkpoint Zip

In [ ]:
if cfg['ZIP_RESULTS_AFTER_RESPONSE_STAGE']:
    with timer.stage('response_checkpoint_zip'):
        checkpoint = write_cot_response_checkpoint_archive(paths=paths, modes=cfg['THINKING_MODES'])
        print('Response checkpoint zip:', checkpoint)
else:
    timer.mark_skipped('response_checkpoint_zip')
    print('Skipping response checkpoint zip')

## 8. Select Analysis Examples

In [ ]:
if (cfg['RUN_OUTLIER_ANALYSIS'] or cfg['RUN_ATTENTION_ANALYSIS']) and selected_ids:
    tokenizer = AutoTokenizer.from_pretrained(cfg['TOKENIZER_NAME'], trust_remote_code=True)
    analysis_eligibility_by_idx = collect_analysis_needle_span_eligibility(
        cfg=cfg,
        paths=paths,
        rows=rows,
        modes=cfg['THINKING_MODES'],
        tokenizer=tokenizer,
    )
else:
    analysis_eligibility_by_idx = None

selected_ids = select_analysis_example_ids(
    rows=rows,
    results_by_mode=results_by_mode,
    max_examples=int(cfg['MAX_ANALYSIS_EXAMPLES']),
    analysis_eligibility_by_idx=analysis_eligibility_by_idx,
)
print('Selected analysis example ids:', selected_ids)
for example_id in selected_ids:
    print(example_id, 'gold=', rows[example_id].get('gold_answer'), 'needles=', len(rows[example_id].get('needles', [])))

## 9. Prepare Mode-Specific Analysis Runs

In [ ]:
if cfg['RUN_OUTLIER_ANALYSIS'] or cfg['RUN_ATTENTION_ANALYSIS']:
    for mode in cfg['THINKING_MODES']:
        with timer.stage(f'prepare_mode_run_{mode}'):
            write_selected_mode_run(
                cfg=cfg,
                paths=paths,
                mode=mode,
                rows=rows,
                selected_ids=selected_ids,
            )
            tokenizer = AutoTokenizer.from_pretrained(cfg['TOKENIZER_NAME'], trust_remote_code=True)
            materialized = ensure_selected_full_sequence_artifacts(
                cfg=cfg,
                paths=paths,
                mode=mode,
                rows=rows,
                selected_ids=selected_ids,
                tokenizer=tokenizer,
            )
            print(f'Prepared {len(materialized)} full-sequence artifact(s) for mode={mode}')
            print(f'Prepared mode={mode} run dir:', paths.mode_dir(mode))
else:
    timer.mark_skipped('prepare_mode_runs')
    print('Skipping mode-specific analysis preparation')

## 10. Hidden-State Capture

In [ ]:
if cfg['RUN_OUTLIER_ANALYSIS'] or cfg['RUN_ATTENTION_ANALYSIS']:
    for mode in cfg['THINKING_MODES']:
        with timer.stage(f'hidden_state_capture_{mode}'):
            model, _tokenizer = load_model_and_tokenizer()
            print_gpu_memory_snapshot(f'after hidden-state model load mode={mode}')
            try:
                hidden_paths = save_selected_hidden_states_for_examples(
                    model=model,
                    mode_dir=paths.mode_dir(mode),
                    example_ids=selected_ids,
                    layers=cfg['LAYERS'],
                    hidden_state_dtype=resolve_analysis_dtype(),
                )
                print(f'Saved {len(hidden_paths)} hidden-state file(s) for mode={mode}')
            finally:
                del model
                release_torch_memory(collect_garbage=True)
else:
    timer.mark_skipped('hidden_state_capture')
    print('Skipping hidden-state capture: analysis disabled or no selected examples')

## 11. Q/K Outlier and Attention-Sink Analysis

In [ ]:
if cfg['RUN_OUTLIER_ANALYSIS'] and selected_ids:
    qk_summaries = {}
    for mode in cfg['THINKING_MODES']:
        with timer.stage(f'qk_outlier_attention_{mode}'):
            summary = run_mode_qk_outlier_analysis(
                cfg=cfg,
                paths=paths,
                mode=mode,
                selected_ids=selected_ids,
            )
            qk_summaries[mode] = summary
            print(json.dumps(summary, indent=2, ensure_ascii=False))
else:
    timer.mark_skipped('qk_outlier_attention')
    qk_summaries = {}
    print('Skipping Q/K outlier analysis: disabled or no selected examples')

## 12. CoT Attention Tables and Plots

In [ ]:
if cfg['RUN_ATTENTION_ANALYSIS'] and selected_ids:
    for mode in cfg['THINKING_MODES']:
        with timer.stage(f'cot_attention_tables_{mode}'):
            table_paths = write_cot_attention_projection_tables(
                cfg=cfg,
                paths=paths,
                mode=mode,
                selected_ids=selected_ids,
            )
            print(f'Wrote {len(table_paths)} CoT attention table(s) for mode={mode}')
        if cfg['RUN_PLOTTING']:
            with timer.stage(f'cot_attention_plots_{mode}'):
                figure_paths = plot_cot_attention_tables(
                    cfg=cfg,
                    paths=paths,
                    mode=mode,
                    selected_ids=selected_ids,
                )
                print(f'Wrote {len(figure_paths)} CoT attention plot(s) for mode={mode}')
else:
    timer.mark_skipped('cot_attention_tables')
    print('Skipping CoT attention tables/plots: disabled or no selected examples')

## 13. Cleanup and Final Archive

In [ ]:
with timer.stage('cleanup_before_archive'):
    removed = cleanup_cot_mode_artifacts(
        paths.run_dir,
        keep_hidden_states=bool(cfg['SAVE_FULL_HIDDEN_STATES']),
        keep_full_sequence_tensors=bool(cfg['SAVE_FULL_HIDDEN_STATES']),
    )
    print('Cleanup removed:', {key: len(value) for key, value in removed.items()})

if cfg['ZIP_RESULTS_AFTER_ANALYSIS_STAGE']:
    with timer.stage('final_archive'):
        archive = archive_cot_results(paths, results_path=cfg['RESULTS_PATH'])
        print('Final archive:', archive)
else:
    timer.mark_skipped('final_archive')
    print('Skipping final archive')